In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch

torch.cuda.empty_cache()
print(torch.cuda.get_device_name(0))       # expect: Tesla

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import json, collections

rows = [json.loads(l) for l in open("train.jsonl") if l.strip()]
tgt  = [json.loads(r["messages"][-1]["content"]) for r in rows]
prompt_keys = {k for r in rows for k in json.loads(r["messages"][1]["content"])}
shapes = collections.Counter(r["shape"] for r in rows)

assert {k for t in tgt for k in t} == {"extracted", "skipped_fields"}, "stale: has next_question"
bad = {k for t in tgt for k in t["extracted"]} - {"property_type", "location", "price", "size_sqft"}
assert not bad, f"stale questionnaire: {bad}"

# The two checks above read only the target. The first v4 run trained on a set carrying
# neither `pending_question` nor the shapes that teach it, and both passed — a field the
# prompt carries has to be checked against the prompt, and a shape against the counts.
assert "pending_question" in prompt_keys, "stale prompt: no pending_question — regenerate"
missing = {"correction", "pending-answer"} - set(shapes)
assert not missing, f"stale shapes: {sorted(missing)} — regenerate"

print(f"{len(rows)} rows — current")       # expect 2250
print("  " + "  ".join(f"{k} {v}" for k, v in shapes.most_common()))

In [ ]:
!pip -q install "transformers>=4.44" "peft>=0.12" accelerate safetensors

In [ ]:
import json, torch
from dataclasses import dataclass
from torch.utils.data import Dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

BASE, IGNORE, MAX_LEN = "Qwen/Qwen2.5-0.5B-Instruct", -100, 1088
tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token


def ids(msgs, *, generation_prompt):
    """Token ids as a flat list.

    Ask for the dict and unwrap it. On transformers 5.x a bare
    ``apply_chat_template(tokenize=True)`` returns a BatchEncoding, whose ``len()`` is the
    number of keys (2) — the prefix check below then rejects every row.
    """
    out = tok.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=generation_prompt, return_dict=True)
    seq = out["input_ids"]
    if seq and isinstance(seq[0], list):        # some versions return a batch of one
        seq = seq[0]
    return list(seq)


class Intake(Dataset):
    def __init__(self, path):
        self.path, self.rows, self.dropped, self.seen = path, [], 0, 0
        for line in open(path, encoding="utf-8"):
            if not line.strip():
                continue
            self.seen += 1
            msgs   = json.loads(line)["messages"]
            full   = ids(msgs, generation_prompt=False)
            prompt = ids(msgs[:-1], generation_prompt=True)
            # The prompt must be a strict prefix, or the mask cuts in the wrong place and
            # the example trains on nothing useful.
            if full[:len(prompt)] != prompt or len(prompt) >= len(full):
                self.dropped += 1
                continue
            full = full[:MAX_LEN]
            if len(prompt) >= len(full):
                self.dropped += 1
                continue
            self.rows.append((full, [IGNORE] * len(prompt) + full[len(prompt):]))

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        ids, lab = self.rows[i]
        return {"input_ids": ids, "labels": lab}


@dataclass
class Collate:
    pad: int

    def __call__(self, f):
        w = max(len(x["input_ids"]) for x in f)
        return {
            "input_ids":      torch.tensor([x["input_ids"] + [self.pad] * (w - len(x["input_ids"])) for x in f]),
            "labels":         torch.tensor([x["labels"]    + [IGNORE]   * (w - len(x["labels"]))    for x in f]),
            "attention_mask": torch.tensor([[1] * len(x["input_ids"]) + [0] * (w - len(x["input_ids"])) for x in f]),
        }


# Matches ml.paths.VAL_PATH, which is `validation.jsonl` — the generator never writes
# `val.jsonl`, so naming it that here fails the upload check below for the wrong reason.
train, val = Intake("train.jsonl"), Intake("validation.jsonl")
print(f"train {len(train)} | val {len(val)} | dropped {train.dropped + val.dropped}")

# Both before the ratio below, which divides by zero on an empty set and buries the cause.
# Nothing-read and everything-rejected are different faults with the same symptom, so they
# get separate checks: no lines read points at the upload, not the encoding.
for ds in (train, val):
    assert ds.seen, f"{ds.path} has no lines — re-upload it, see step 2"
assert train.dropped == 0, "prompt was not a prefix — chat template changed?"

sup = sum(sum(1 for x in lab if x != IGNORE) for _, lab in train.rows)
tot = sum(len(lab) for _, lab in train.rows)
print(f"supervised tokens: {sup}/{tot} ({sup / tot:.1%})")
assert 0.02 < sup / tot < 0.15, "loss mask looks wrong"

model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.bfloat16, device_map="cuda")
model.config.use_cache = False
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()          # ~0.9% — 100% means LoRA did not attach
model.enable_input_require_grads()          # gradient checkpointing needs this under PEFT

Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="lora-intake-v4", per_device_train_batch_size=2,
        gradient_accumulation_steps=4, per_device_eval_batch_size=2,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        num_train_epochs=2, learning_rate=2e-4,
        lr_scheduler_type="cosine", warmup_ratio=0.03, logging_steps=10,
        eval_strategy="steps", eval_steps=50, save_strategy="epoch",
        bf16=True, report_to=[], seed=17, remove_unused_columns=False),
    train_dataset=train, eval_dataset=val, data_collator=Collate(tok.pad_token_id),
).train()

print(f"peak GPU: {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")

model.save_pretrained("lora-intake-v4")
tok.save_pretrained("lora-intake-v4")
!zip -qr lora-intake-v4.zip lora-intake-v4 && ls -lh lora-intake-v4.zip

In [ ]:
files.download("lora-intake-v4.zip")       # ~17 MB